[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/robot_arm/robot_arm.ipynb)

# Robot Arm Kinematics

This example computes the forward kinematics, differential velocity, static joint torques, and inverse-kinematics trajectory tracking of an articulated robot arm with a yawing base, a pitching elbow, and a pitching wrist. Forward kinematics places the links and end effector from the joint angles, differential kinematics relates joint angular rates to tip velocity and external loads to static joint torques, and iterative inverse kinematics steers the joints so the tip homes onto a closed spatial loop and tracks a moving target around it.

In three-dimensional projective geometric algebra, each joint axis is modelled as a line bivector met from two planes, and scaling that axis by the joint angle gives a twist whose half-angle exponential `(joints * 0.5).exp()` is the joint motor. Composing those motors on the right places each link in the frame accumulated from earlier joints, and sandwiching the home-pose axes `frames >> axes` carries the joint lines into the current configuration so the world twist moves the tip through `twist.commutator(tip)`. The term ***forque*** here means the line-bound join `load_point & load_force` of an application point with a force direction ([Gunn, 2011](https://arxiv.org/pdf/1101.4542); [Dorst and De Keninck, 2022](https://bivector.net/PGAdyn.pdf)), and pairing the carried axes `(frames >> axes) & forque` gives the static joint torques directly while `(frames >> axes).commutator(tip).lstsq(target - tip)` updates the joints toward a target.

In matrix robotics notation, forward kinematics chains $4\times 4$ homogeneous transforms $T_i = \prod_{k \le i} \begin{bmatrix} R_k(q_k) & \mathbf p_k \\ \mathbf 0^{\mathsf T} & 1 \end{bmatrix}$, each column of the $6\times n$ geometric Jacobian stacks a cross-product linear velocity above an angular axis as $J_i(\mathbf q) = \begin{bmatrix} \mathbf z_i \times (\mathbf p_{\text{tip}} - \mathbf o_i) \\ \mathbf z_i \end{bmatrix}$ with $\mathbf z_i = R_{i-1}\hat{\mathbf z}_i$ so $\mathbf v_{\text{tip}} = J_v(\mathbf q)\dot{\mathbf q}$, and shifting a force $\mathbf f$ at $\mathbf r$ into a reference wrench $\mathbf w = \begin{bmatrix} \mathbf f \\ (\mathbf r - \mathbf p_{\text{tip}}) \times \mathbf f \end{bmatrix}$ yields static joint torques $\boldsymbol\tau = J(\mathbf q)^{\mathsf T}\mathbf w$.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, concatenate, stack
from numga.algebras import PGA3D as ga
from examples.util.animation import show_states
from examples.mechanics.robot_arm import render, scenarios

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Point = ga.gatype.antivector()                      # Antivector-
# A joint axis scaled by an angle or rate forms an extrinsic twist, while a line-bound load forms an intrinsic forque.
Twist = ga.gatype.bivector()                        # Bivector+
Forque = ga.gatype.antibivector()                   # Antibivector-
Motor = ga.gatype.rotor()                           # Rotor+
Scalar = ga.gatype.scalar()                         # Scalar+

## Joint axes and forward kinematics

Each joint axis in the home pose is the meet of two planes, giving a vertical yaw line through the base and horizontal pitch lines at the elbow and wrist heights. Scaling each axis by its angle forms the joint twist, and because each joint turns in the frame left by the joints before it, right-multiplying the joint motors `(joints * 0.5).exp()` equals reversing the cumulative product of their reverses.

In [ ]:
elbow_height, wrist_height, tip_height = scenarios.elbow_height, scenarios.wrist_height, scenarios.tip_height
rest_angles = scenarios.rest_angles

In [ ]:
def forward_kinematics(joints: Twist) -> tuple[Motor, Motor]:
    """Right-multiplication of the joint motors (`[...] Motor+`) gives the tip motor (`[] Motor+`) and pre-joint frames (`[...] Motor+`)."""
    steps = (joints * 0.5).exp()                                               # [...] Motor+
    # Right-multiplying sequential joint steps equals reversing the cumulative product of their reverses.
    poses = steps.reverse().cumprod(axis=0).reverse()                          # [...] Motor+
    frames = concatenate([mv.rotor().reshape(1), poses[:-1]])                  # [...] Motor+
    return poses[-1], frames


# Base yaw turns about the vertical z-axis, while elbow and wrist pitch about horizontal lines at their heights.
axes = stack([
    -mv.xy,
    mv.x ^ (mv.z - elbow_height * mv.w),
    mv.x ^ (mv.z - wrist_height * mv.w),
])                                                                             # [3] Twist+
tip_home = mv.point([0, 0, tip_height])                                        # [] Point-
rest = axes * rest_angles                                                      # [3] Twist+
pose, frames = forward_kinematics(rest)                                        # [] Motor+, [3] Motor+
rest_tip = pose >> tip_home                                                    # [] Point-
print("rest tip:\n", rest_tip.coefficient_table())

## Tip velocity and static joint torques

Sandwiching each joint rate into its pre-joint frame and summing produces the spatial twist of the distal link, whose commutator `twist.commutator(tip)` gives the instantaneous tip velocity. Pairing each carried axis `(frames >> axes) & forque` with an external load forque measures the virtual work per unit angle of that joint, giving the torque that the load exerts about each joint.

In [ ]:
joint_rates = scenarios.joint_rates
load_point, load_force = scenarios.load_point, scenarios.load_force

In [ ]:
def mechanics(
    joints: Twist,
    axes: Twist,
    rates: Twist,
    tip_home: Point,
    forque: Forque,
) -> tuple[Point, Scalar]:
    """Carried joint rates and axes determine the tip velocity (`[] Point-`) and static joint torques (`[...] Scalar+`) at `joints`."""
    pose, frames = forward_kinematics(joints)                                  # [] Motor+, [...] Motor+
    tip = pose >> tip_home                                                     # [] Point-

    # Carrying each joint rate into its frame and summing forms the spatial twist that moves the tip.
    twist = (frames >> rates).sum(axis=0)                                      # [] Twist+
    velocity = twist.commutator(tip)                                           # [] Point-

    # Pairing each carried joint axis with the applied forque reads the work per unit joint angle.
    joint_torques = (frames >> axes) & forque                                  # [...] Scalar+
    return velocity, joint_torques


rates = axes * joint_rates                                                     # [3] Twist+
forque = mv.direction(load_force) & mv.point(load_point)                       # [] Forque-
velocity, joint_torques = mechanics(rest, axes, rates, tip_home, forque)       # [] Point-, [3] Scalar+
print("tip velocity:\n", velocity.coefficient_table())
print("joint torques:", joint_torques.to_array())

## Inverse kinematics and loop tracking

At any configuration, the commutators `(frames >> axes).commutator(tip)` give the tip velocities generated by unit rates on each joint. Fitting the remaining displacement `target - tip` in least squares updates the joint twists at each Gauss-Newton step, homing the arm from rest onto a closed saddle loop before tracking the moving target around the path.

In [ ]:
n_homing_steps, n_tracking_steps = scenarios.n_homing_steps, scenarios.n_tracking_steps
n_frames = scenarios.n_frames
targets = scenarios.loop_targets(n_frames)                                     # [n_frames] Point-
boxes_home = scenarios.link_boxes(scenarios.n_links, scenarios.link_width, scenarios.link_depth)   # [3, 8] Point-

In [ ]:
def inverse_kinematics(
    joints: Twist,
    axes: Twist,
    tip_home: Point,
    target: Point,
    n_steps: int,
) -> tuple[Twist, Motor, Point]:
    """Gauss-Newton updates (`n_steps`) drive `joints` (`[...] Twist+`) toward `target`, returning the solved joints, link motors, and tip."""
    for _ in range(n_steps):
        pose, frames = forward_kinematics(joints)                              # [] Motor+, [...] Motor+
        tip = pose >> tip_home                                                 # [] Point-
        # Unit rates along the carried joint axes induce the basis tip velocities for the linear step.
        columns = (frames >> axes).commutator(tip)                             # [...] Point-
        joints = joints + axes * columns.lstsq(target - tip)                   # [...] Twist+
    pose, frames = forward_kinematics(joints)                                  # [] Motor+, [...] Motor+
    link_motors = concatenate([frames[1:], pose.reshape(1)])                   # [...] Motor+
    return joints, link_motors, pose >> tip_home


def tracking_states(
    joints: Twist,
    axes: Twist,
    tip_home: Point,
    targets: Point,
    n_steps: int,
) -> Iterator[tuple[Twist, Motor, Point, Point]]:
    """Yields the solved joints, link motors, target, and tip across a sequence of `targets` (`[...] Point-`)."""
    for target in targets:
        joints, link_motors, tip = inverse_kinematics(joints, axes, tip_home, target, n_steps)
        yield joints, link_motors, target, tip


homed_joints, _, homed_tip = inverse_kinematics(rest, axes, tip_home, targets[0], n_homing_steps)   # [3] Twist+, [3] Motor+, [] Point-
homed_angles = (homed_joints | axes) / (axes | axes)                           # [3] Scalar+
print("homed joint angles (rad):", homed_angles.to_array())
scene_stream = (
    (link_motors[:, None] >> boxes_home, step_target, step_tip)
    for _, link_motors, step_target, step_tip in tracking_states(homed_joints, axes, tip_home, targets, n_tracking_steps)
)
tracked_states = show_states(
    scene_stream,
    render.tracking_frames,
    name="robot_arm",
    duration_ms=scenarios.frame_duration_ms,
    n_frames=n_frames,
)

The translucent blue, cyan, and purple boxes show the base, elbow, and wrist links turning smoothly as the end effector follows the red target sphere and sweeps out the closed red trail.

In [ ]:
# checks
assert (pose - (rest[0] * 0.5).exp() * (rest[1] * 0.5).exp() * (rest[2] * 0.5).exp()).scalar_norm_squared() < 1e-10
assert ((mv.w & rest_tip) - 1).abs() < 1e-8
assert (mv.w & velocity).abs() < 1e-8
assert ((joint_torques * joint_rates).sum() - ((frames >> rates).sum(axis=0) & forque)).abs() < 1e-8
assert ((joint_torques * joint_rates).sum() - (frames >> rates).sum(axis=0).commutator(mv.point(load_point)).dot(mv.direction(load_force))).abs() < 1e-8
assert (targets[0] & homed_tip).norm() < 1e-6
tracked_tips = stack([step_tip for _, _, step_tip in tracked_states])          # [n_frames] Point-
assert len(tracked_states) == n_frames
assert ((targets & tracked_tips).norm() < 1e-4).all()